In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import math
from pgpwdata import PGPWData
import plotly.graph_objects as go
import plotly
import datetime

In [ ]:
class DataLoader:
    def __init__(self, cvs_path_or_df, name_of_date_column, name_of_Q_column):
        self.cvs_path_or_df = cvs_path_or_df
        self._name_of_date_column = name_of_date_column
        self._name_of_Q_column = name_of_Q_column
        self.df = self.load_data()
    
    def load_data(self):
        if isinstance(self.cvs_path_or_df, pd.DataFrame):
            self._df = self.cvs_path_or_df.copy()  # Set the DataFrame directly
            # Convert columns to numeric, handling errors by setting non-numeric values to NaN
            if isinstance(self._df[self._name_of_Q_column][0], str):
                self._df[self._name_of_Q_column] = pd.to_numeric(self._df[self._name_of_Q_column], errors='coerce')

        elif isinstance(cvs_path_or_df, str):
            try:
                print(f"Importing CSV with the file path: {csv_path}")
                self._df = pd.read_csv(cvs_path_or_df)
                # Convert columns to numeric, handling errors by setting non-numeric values to NaN
                if isinstance(self._df[self._name_of_Q_column][0], str):
                    self._df[self._name_of_Q_column] = pd.to_numeric(self._df[self._name_of_Q_column], errors='coerce')
                # self._df['Year'] = pd.to_numeric(self._df['Year'], errors='coerce')
            except Exception as e:
                print(f"Error reading CSV file: {e}")
                
        else:
            print("Invalid input type. Please provide either a pandas DataFrame or a CSV file path.")
            


In [ ]:
class StatisticsCalculator:
    def __init__(self, DataLoader):
        self.data_loader = DataLoader
        self._df = self.data_loader._df
        self._df['Date'] = pd.to_datetime(self._df[self.data_loader._name_of_date_column])
        self._df = self._df[~self._df.duplicated('Date')]
        self._df['month'] = self._df['Date'].dt.month
        self._df['Year'] = self._df['Date'].dt.year
        self._df['month-day'] = self._df['Date'].apply(lambda x: x.strftime('%m-%d'))
        self._df['Water Year'] = self._df['Date'].dt.year.where(self._df['Date'].dt.month<10, self._df['Date'].dt.year+1)
        self.stats = self.calculate_statistics()
    
    def calculate_statistics(self, **kwargs):

        self._stats = self._df.groupby("month-day")[self.data_loader._name_of_Q_column].agg(['mean', 'median', 'std', ("q25", lambda x: x.quantile(0.25)), ("q75", lambda y: y.quantile(0.75))])
        self._monthly_stats = self._df.groupby("month")[self.data_loader._name_of_Q_column].agg(['mean', 'median', 'std', ("q25", lambda x: x.quantile(0.25)), ("q75", lambda y: y.quantile(0.75))])
        self._mean = self._stats.iloc[:, 0]
        self._median = self._stats.iloc[:, 1]
        self._st_dev = self._stats.iloc[:, 2]
        
        
#         minData = [np.nanmin(a) for a in statsData]
#         maxData = [np.nanmax(a) for a in statsData]
#         meanData = [np.nanpercentile(a, 50) for a in statsData]
#         lowestData = [np.nanpercentile(a, 10) for a in statsData]
#         highestData = [np.nanpercentile(a, 90) for a in statsData]
#         lowData = [np.nanpercentile(a, 30) for a in statsData]
#         highData = [np.nanpercentile(a, 70) for a in statsData]
        
        
        self._percentile25 = self._stats.iloc[:, 3]
        self._percentile75 = self._stats.iloc[:, 4]
        self._lower_bound_st_dev = self._mean - self._st_dev
        self._upper_bound_st_dev = self._mean + self._st_dev
        self._lower_bound_percentile25 = self._mean - self._percentile25
        self._upper_bound_percentile75 = self._mean + self._percentile75

        water_year = kwargs.get('water_year', True)
        print(f'water_year: {water_year}')
        year = 'Water Year' if water_year else 'Year'
        start_year = kwargs.get("start_year", self._df[year].iloc[0])
        end_year = kwargs.get("end_year", self._df[year].iloc[-1])
        self._df = self._df[(self._df[year] >= start_year) & (self._df[year] <= end_year)]
        self._unique_years = self._df[year].unique()
        self._start_year, self._end_year = self._unique_years[0], self._unique_years[-1]
        self._num_of_decades = math.ceil((self._end_year - self._start_year) / 10)
        self._unique_decades = self._df[year].apply(lambda year: (year // 10) * 10).unique()

        # Convert years in _pivot_table to water years if necessary

        self._pivot_table = self._df.copy()
        self._pivot_table['month-day'] = self._pivot_table['Date'].apply(lambda x: x.strftime('%m-%d'))
        self._pivot_table = self._pivot_table.pivot(index="month-day", columns=year, values=self.data_loader._name_of_Q_column)
        
        if water_year == True:
            self._pivot_table = self._pivot_table[self._df['Water Year'].unique()]
        else:
            self._pivot_table = self._pivot_table[self._df['Year'].unique()]
            
        self._pivot_table_monthly = self._df.pivot(columns='month', values=self.data_loader._name_of_Q_column)
        self._pivot_table_yearly_stats = {year: self._pivot_table.iloc[:, i].describe() for i, year in enumerate(self._pivot_table.columns)}

        # Call _prepare_data_for_plotting here
        # self._prepare_data_for_plotting(start_year, end_year, water_year=water_year)

    ##  KEEP!!!!!!  And fix up. - calculates volume under curve/water supply for the year
    # def calculate_yearly_volumes(self):
    #     years = []
    #     area = []
    #     for year in self._pivot_table.columns:
    #         dates = list(range(0, len(self._pivot_table[year].dropna())))
    #         area.append(trapz(self._pivot_table[year].dropna(), dates))
    #         years.append(year)

    #     Area_dict = OrderedDict()
    #     for key, value in zip(years, area):
    #         Area_dict[key] = value
        # return Area_dict

    

In [ ]:
# class PlotUtils:
#     def __init__(self, StatisticsCalculator):
#         self._StatisticsCalculator = StatisticsCalculator
#         self._colors = ['crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod", 'crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod"]


#     def _plot_central_tendency_stats(self, ax, plot_stats):
#         if plot_stats:
#             self._StatisticsCalculator._mean.plot(ax=ax, label="Mean", linestyle=':', color='black', linewidth=1.5, zorder=3)
#             self._StatisticsCalculator._median.plot(ax=ax, label="Median", linestyle=':', color='red', linewidth=1.5, zorder=3)

#     def _plot_highlighted_years(self, ax, highlight_years):
#         for i, year in enumerate(highlight_years):
#             # if water_year:
#             #     year = 'Water Year'
#             # else:
#             #     year = 'Year'

#             self._StatisticsCalculator._pivot_table[year].plot(ax=ax, linewidth=1.6, zorder=3, color=self._colors[i])

#     def _plot_quartile_shading(self, ax, quartile_shading, quartile_shading_alpha, kwargs):
       

#         if quartile_shading:
#             zorder = kwargs.get('quartile_shading_zorder', 1)

#             plt.fill_between(
#                 list(range(0, len(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0]))),
#                 pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float),
#                 pd.DataFrame(self._StatisticsCalculator._lower_bound_st_dev).iloc[:, 0].astype(float),
#                 where=(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float) > pd.DataFrame(self._StatisticsCalculator._lower_bound_percentile25).iloc[:, 0].astype(float)),
#                 interpolate=True, color='yellow', alpha=quartile_shading_alpha, zorder=zorder, label="q25-q75")

#             plt.fill_between(
#                 list(range(0, len(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0]))),
#                 pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float),
#                 pd.DataFrame(self._StatisticsCalculator._upper_bound_st_dev).iloc[:, 0].astype(float),
#                 where=(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float) < pd.DataFrame(self._StatisticsCalculator._upper_bound_percentile75).iloc[:, 0].astype(float)),
#                 interpolate=True, color='yellow', alpha=quartile_shading_alpha, zorder=zorder)

#     def _plot_grouped_by_decade(self, ax, kwargs):
#         for i, decade in enumerate(self._StatisticsCalculator._unique_decades):
#             years_in_decade = [year for year in self._StatisticsCalculator._unique_years if decade <= year < decade + 10]
#             mean_values = self._StatisticsCalculator._pivot_table[years_in_decade].mean(axis=1)
#             std_dev_values = self._StatisticsCalculator._pivot_table[years_in_decade].std(axis=1)
#             ax.plot(self._StatisticsCalculator._pivot_table.index, mean_values, label=f'Decade {decade}s', color=self._colors[i])
#             ax.fill_between(self._StatisticsCalculator._pivot_table.index, mean_values - std_dev_values, mean_values + std_dev_values, alpha=0.2, color=self._colors[i])

#     def _plot_individual_years(self, ax, series_alpha, kwargs):
#         for i, year in enumerate(self._StatisticsCalculator._unique_years):
#             ax.plot(self._StatisticsCalculator._pivot_table.index, self._StatisticsCalculator._pivot_table[year], label=f'{year}', alpha=series_alpha)

#     def _customize_plot(self, ax, kwargs):
#         self._forced_x_positions = kwargs.get('forced_x_positions', [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336]),
#         self._forced_x_labels = kwargs.get('forced_x_labels', ['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01', '11-01', '12-01'])

#         if self._forced_x_positions is not None and self._forced_x_labels is not None:
# #             print(self._forced_x_positions[0])
# #             print(self._forced_x_labels)
#             ax.set_xticks(self._forced_x_positions[0])
#             ax.set_xticklabels(self._forced_x_labels, rotation=45)
#             xlim_min = self._forced_x_positions[0][0]
#             xlim_max = self._forced_x_positions[0][-1]
#             ax.set_xlim([xlim_min, xlim_max])
#             ax.set_ylim([kwargs.get('y_lower_lim', self._StatisticsCalculator._df[self._StatisticsCalculator.data_loader._name_of_Q_column].min()), kwargs.get('y_upper_lim', self._StatisticsCalculator._df[self._StatisticsCalculator.data_loader._name_of_Q_column].max())])

#         plt.grid(color='green', linestyle=":", linewidth=0.5)
#         plt.xlabel('Month-Day')
#         plt.ylabel(kwargs.get('ylabel', "Discharge (cfs)"))
#         plt.title(kwargs.get('title'))

#         if kwargs.get('legend_mode') == "partial":
# #             labels = ['Mean', 'Median', 'q25-q75'] + kwargs.get('highlight_years', [])
#             plt.legend(loc=kwargs.get('legend_pos', 'upper right'), ncol=kwargs.get('legend_ncol', 1), labels=['Mean', 'Median'] + kwargs.get('highlight_years', []) + ['q25-q75'])
#         else:
#             plt.legend(loc=kwargs.get('legend_pos'), ncol=kwargs.get('legend_ncol'))#, labels=labels)


In [ ]:
# class StackedLinePlots:
#     '''
#     Keyword Arguments and defaults include:
#         plot_central_tendency_stats=True,
#         highlight_years=[],
#         water_year=True,
#         quartile_shading=True,
#         quartile_shading_alpha=0.5,
#         group_by_decade=False,
#         series_alpha=0.3,
#         quartile_shading_zorder=1,
#         'forced_x_positions'=[1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336],
#         'forced_x_labels'=['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01', '11-01', '12-01']
#         'y_lower_lim'=0,
#         'y_upper_lim'=25,
#         'ylabel'='Discharge',
#         'title',
#         'legend_mode'='partial',
#         'legend_pos'='upper right',
#         'legend_ncol'=1
#     '''
#     def __init__(self, StatisticsCalculator, **kwargs):
#         self._StatisticsCalculator = StatisticsCalculator
#         self._PlotUtils = PlotUtils(StatisticsCalculator)
            
#         # self._prepare_data_for_plotting(kwargs.get('input_start_year', 2010), kwargs.get('input_end_year', 2020))

#         fig, ax = plt.subplots(figsize=(9, 7))

#         self._PlotUtils._plot_central_tendency_stats(ax, kwargs.get('plot_central_tendency_stats', True))
#         self._PlotUtils._plot_highlighted_years(ax, kwargs.get('highlight_years'))
#         self._PlotUtils._plot_quartile_shading(ax, kwargs.get('quartile_shading', True), kwargs.get('quartile_shading_alpha', 0.5), kwargs)

#         if kwargs.get('group_by_decade', False):
#             self._PlotUtils._plot_grouped_by_decade(ax, kwargs)
#         else:
#             self._PlotUtils._plot_individual_years(ax, kwargs.get('series_alpha', 0.3), kwargs)

#         self._PlotUtils._customize_plot(ax, kwargs)
#         plt.show(kwargs.get('title'))

## TESTING

In [ ]:
#Issues:
##legend labels don't match up with actual name of label
##better understand **kwargs.  Can't set any of the kwargs in CalculateStatistics. They just run to defaults.  Same in the other classe?
##need to organize data with water year
##fig out qwarks with highlight_years
## add ability to change y axis label

## Plotly

In [ ]:
import plotly.express as px
import datetime

class PlotUtilsPlotly:
    def __init__(self, StatisticsCalculator):
        self._StatisticsCalculator = StatisticsCalculator
        self._colors = px.colors.qualitative.Plotly #['crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod", 'crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod"]

    def _plot_central_tendency_stats(self, fig, plot_stats):
        if plot_stats:
            fig.add_trace(go.Scatter(x=self._StatisticsCalculator._df.index, y=self._StatisticsCalculator._mean,
                                     mode='lines', name="Mean", line=dict(color='black', dash='dash')))
            fig.add_trace(go.Scatter(x=self._StatisticsCalculator._df.index, y=self._StatisticsCalculator._median,
                                     mode='lines', name="Median", line=dict(color='black', dash='dot')))
 
    def _plot_highlighted_years(self, fig, highlight_years):
        for i, year in enumerate(highlight_years):
            fig.add_trace(go.Scatter(x=self._StatisticsCalculator._df.index, y=self._StatisticsCalculator._pivot_table[year],
                                     mode='lines', name=str(year), line=dict(color=self._colors[i], width=2.5), opacity=1, hoverinfo='text'))
 
    def _plot_quartile_shading(self, fig, quartile_shading, quartile_shading_alpha, kwargs):
        if quartile_shading:
            for i, row in self._StatisticsCalculator._pivot_table.iterrows():
                fig.add_trace(go.Scatter(x=row.index, y=row,
                                         fill='tonexty',
                                         fillcolor='yellow',
                                         line=dict(color='rgba(255,255,255,0)'),
                                         name="q25-q75", opacity=quartile_shading_alpha))

    def _plot_grouped_by_decade(self, fig, kwargs):
        for decade in self._StatisticsCalculator._unique_decades:
            years_in_decade = [year for year in self._StatisticsCalculator._unique_years if decade <= year < decade + 10]
            mean_values = self._StatisticsCalculator._pivot_table[years_in_decade].mean(axis=1)
            std_dev_values = self._StatisticsCalculator._pivot_table[years_in_decade].std(axis=1)
            fig.add_trace(go.Scatter(x=self._StatisticsCalculator._pivot_table.index, y=mean_values,
                                     mode='lines', name=f'Decade {decade}s', line=dict(color=self._colors[i])))
            fig.add_trace(go.Scatter(x=self._StatisticsCalculator._pivot_table.index, y=mean_values + std_dev_values,
                                     fill='tonexty',
                                     fillcolor=self._colors[i],
                                     line=dict(color='rgba(255,255,255,0)'),
                                     name="Decade std dev", opacity=0.2))
    def _plot_vline(self, fig):
        current_monthday = datetime.datetime.now().strftime('%m%d')
        print(datetime.datetime.now().strftime('%j'))
#         print(f'{current_monthday[:2]}-{current_monthday[2:]}')
#         fig.add_vline(x=f'{current_monthday[:1]}-{current_monthday[2:]}', line_dash='dash', line_color='red')#, annotated_text='Current Date')
        fig.add_vline(x=datetime.datetime.now().strftime('%j'), line_dash='dash', line_color='red')#, annotated_text='Current Date')

    def _plot_individual_years(self, fig, series_alpha, kwargs):
        for year in self._StatisticsCalculator._unique_years:
            fig.add_trace(go.Scatter(x=self._StatisticsCalculator._pivot_table.reset_index().index, 
                                     y=self._StatisticsCalculator._pivot_table[int(year)],
                                     mode='lines', name=str(year), opacity=series_alpha, showlegend=False, hoverinfo='none'))
             
    def _customize_plot(self, fig, kwargs):
        fig.update_layout(xaxis=dict(
            tickvals=kwargs.get('forced_x_positions', [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336]),
            ticktext=kwargs.get('forced_x_labels', ['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01', '11-01', '12-01']),
            tickangle=45),
            yaxis=dict(range=[kwargs.get('y_lower_lim', self._StatisticsCalculator._df[self._StatisticsCalculator.data_loader._name_of_Q_column].min()),
                                kwargs.get('y_upper_lim', self._StatisticsCalculator._df[self._StatisticsCalculator.data_loader._name_of_Q_column].max())]),
            title=kwargs.get('title', ''),
            legend=dict(x=0, y=1),
            plot_bgcolor='rgba(0,0,0,0)',
            paper_bgcolor='rgba(0,0,0,0)',
              )


In [ ]:
# fig = go.Figure()
# for year in s._unique_years:
#             fig.add_trace(go.Scatter(x=pd.DataFrame(s._pivot_table[int(year)]).reset_index().index, y=pd.DataFrame(s._pivot_table[int(year)]).reset_index()[int(year)],
#                                      mode='lines', name=str(year), opacity=1))
        
# fig.add_vline(x=150, line_dash='solid', line_color='red')#, annotated_text='Current Date')

# # fig.add_trace(go.Scatter(x=s._pivot_table[int(1950)].index, y=s._pivot_table[int(2010)],
# #                                      mode='lines', name=str(1950), opacity=1))
# fig.show()


In [ ]:
# pd.DataFrame(s._pivot_table[int('2012')]).reset_index()

In [ ]:
class StackedLinePlotsPlotly:
    '''
    Keyword Arguments and defaults include:
        plot_central_tendency_stats=True,
        highlight_years=[],
        water_year=True,
        quartile_shading=True,
        quartile_shading_alpha=0.5,
        group_by_decade=False,
        series_alpha=0.3,
        quartile_shading_zorder=1,
        'forced_x_positions'=[1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336],
        'forced_x_labels'=['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01', '11-01', '12-01']
        'y_lower_lim'=0,
        'y_upper_lim'=25,
        'ylabel'='Discharge',
        'title',
        'legend_mode'='partial',
        'legend_pos'='upper right',
        'legend_ncol'=1
    '''
    def __init__(self, StatisticsCalculator, **kwargs):
        self._StatisticsCalculator = StatisticsCalculator
        self._PlotUtils = PlotUtilsPlotly(StatisticsCalculator)
           
        fig = go.Figure()
        
        if kwargs.get('group_by_decade', False):
            self._PlotUtils._plot_grouped_by_decade(fig, kwargs)
        else:
            self._PlotUtils._plot_individual_years(fig, kwargs.get('series_alpha', 1), kwargs)
 
        self._PlotUtils._customize_plot(fig, kwargs)
    
        self._PlotUtils._plot_central_tendency_stats(fig, kwargs.get('plot_central_tendency_stats', True))
        self._PlotUtils._plot_highlighted_years(fig, kwargs.get('highlight_years'))
        self._PlotUtils._plot_quartile_shading(fig, kwargs.get('quartile_shading', True), kwargs.get('quartile_shading_alpha', 0.5), kwargs)
        self._PlotUtils._plot_vline(fig)
        self.fig = fig
        fig.show()
#         plotly.offline.plot(fig, filename=f'{}.html')


In [ ]:
# a = DataLoader(SanJuan_df, 'dateTime', name_of_Q_column='value')
# s = StatisticsCalculator(a)
# StackedLinePlotsPlotly(s,
#                 title="San Juan River",
#                 water_year=True,
# #                 group_by_decade=True,
#                 highlight_years=[2024],
#                 quartile_shading=False, 
# #                 quartile_shading_alpha=0.5,
#                 series_labels=False,
#                 series_alpha=.15,
# #                 y_upper_lim=110000, 
#                 legend_pos="best",
#                 legend_mode="partial",
#                 legend_ncol=1,)

In [ ]:
data = PGPWData.getUSGSdata(start_date='1990-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=12413500).reset_index()
a = DataLoader(data, 'dateTime', name_of_Q_column='value')
s = StatisticsCalculator(a)
StackedLinePlotsPlotly(s,
                title=site[0],
                 highlight_years=[2024],
                 quartile_shading=False, 
                 quartile_shading_alpha=0.5,
                 series_labels=False,
                 series_alpha=.10,
                 y_upper_lim=25000,
                 y_lower_lim=0,
                 legend_pos="best",
                 legend_mode="partial",
                 legend_ncol=1,)

In [ ]:
s._median

In [ ]:
vip_sites = {

    #Upper Tribs
    'Okanogan River Near Tonasket, WA': 12445000,
    'Chelan River at Chelan': 12452500,
    #Yakima
    'Yakima at Kiona': 12510500,
    'Yakima River at Umtanum, WA': 12484500,
    #Pend Orielle
    'Clark Fork below Missoula MT': 12353000,
    'Clark Fork at St. Regis MT': 12354500,
    'Flathead at Columbia Falls MT': 12363000,
    #Spokane/Coeur D Alene
    'Spokane River near Post Falls, ID':  12419000,
    'Coeur D Alene River nr Cataldo, ID': 12413500,
    # Clearwater and Salmon
    'Clearwater River at Orofino ID': 13340000,      
    'MF Salmon River at Mouth Nr Shoup, ID': 13310199,
    #Snake River
        #Upper/headwaters
        #Mid/Boise
        #Lower
    'Snake River at Anatone': 13334300,
#     'Grande Ronde River at Troy, OR': 13333000,  #doesn't have 2024 data and causes script to crash...prob add error handling
#     'Snake River at Hells Canyon Dam': 13290450,
    'Snake River at Weiser ID': 13269000, 
    #Interior OR Basins
    'Owyhee River nr Rome OR': 13181000,
    'John Day at McDonald Ferry': 14048000,
    'John Day at Service Creek': 14046500,
    'Deschutes at Moody': 14103000,
    'Umatilla nr Pendleton': 14020850,
    #Cascades
    'Klickitat': 14113000,
    'Hood River': 14120000,
    'White Salmon': 14123500,
    'Walla Walla nr Touchet': 14018500,
}

In [ ]:
# USGS_sites = {
#       'John Day at McDonald Ferry': 14048000,
#       'John Day at Service Creek': 14046500,
#       'Deschutes at Moody': 14103000,
#       'Umatilla nr Pendleton': 14020850,
#       'Klickitat': 14113000,
#       'Hood River': 14120000,
#       'White Salmon': 14123500,
#       'Chelan River at Chelan': 12452500,
#       'Walla Walla nr Touchet': 14018500,
#       'Yakima at Kiona': 12510500
#        }

In [ ]:
for site in vip_sites.items():
    print(site)
    data = PGPWData.getUSGSdata(start_date='1990-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=str(site[1])).reset_index()
    a = DataLoader(data, 'dateTime', name_of_Q_column='value')
    s = StatisticsCalculator(a)
    plot = StackedLinePlotsPlotly(s,
                    title=site[0],
                     highlight_years=[2024],
                     quartile_shading=False, 
                     quartile_shading_alpha=0.5,
                     series_labels=False,
                     series_alpha=.10,
                     y_upper_lim=25000,
                     y_lower_lim=0,
                     legend_pos="best",
                     legend_mode="partial",
                     legend_ncol=1,)
    
    with open('historicalplots_plotly.html', 'a') as f:
        f.write(plot.fig.to_html(full_html=False, include_plotlyjs='cdn'))


In [ ]:
Spokane_sites = {
    "CDA River near Cataldo": 12413500,
    "St. Joe River at Red Ives Ranger Station, ID": 12413875,
    "Spokane River nr Post Falls, ID": 12419000
}

In [ ]:
for site in Spokane_sites.items():
    print(site)
    data = PGPWData.getUSGSdata(start_date='1900-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=str(site[1])).reset_index()
    a = DataLoader(data, 'dateTime', name_of_Q_column='value')
    s = StatisticsCalculator(a)
    StackedLinePlotsPlotly(s,
                    title=site[0],
                     highlight_years=[2024],
                     quartile_shading=False, 
                     quartile_shading_alpha=0.5,
                     series_labels=False,
                     series_alpha=.10,
                     y_upper_lim=25000,
                     y_lower_lim=0,
                     legend_pos="best",
                     legend_mode="partial",
                     legend_ncol=1,)

In [ ]:
Snake_sites = {
    'Snake River at Anatone': 13334300,
    'Grande Ronde River at Troy, OR': 13333000,
    'Salmon at White Bird': 13317000,
    'Snake River at Hells Canyon Dam': 13290450
}

Green_River_UT = {
    'Green River at Green River, UT': '09315000',
    'Green River Near Greendale, UT': '09234500'
}

In [ ]:
for site in Snake_sites.items():
    print(site)
    data = PGPWData.getUSGSdata(start_date='1900-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=str(site[1])).reset_index()
    a = DataLoader(data, 'dateTime', name_of_Q_column='value')
    s = StatisticsCalculator(a)
    StackedLinePlotsPlotly(s,
                    title=site[0],
                     highlight_years=[2024],
                     quartile_shading=False, 
                     quartile_shading_alpha=0.5,
                     series_labels=False,
                     series_alpha=.10,
                     y_upper_lim=25000,
                     y_lower_lim=0,
                     legend_pos="best",
                     legend_mode="partial",
                     legend_ncol=1,)

In [ ]:
s._df

In [ ]:
s._pivot_table#.sample(40)

In [ ]:
water_year_categories = pd.CategoricalDtype(
    categories=[str(i) for i in range(10,13)] + [str(i) for i in range (1,10)],
    ordered=True)

In [ ]:
water_year_categories

In [ ]:
s._df

In [ ]:
# s._df['sort_month'] = s._df['month'].apply(lambda x: {'Oct':1, 'Nov':2, 'Dec':3, 'Jan':4,, 'Feb':5, 'Mar':6, 'Apr':7, 'May':8, "Jun":9, "Jul":10, 'Aug':11, 'Sep':12}[x])
s._df['month'] = s._df['month'].astype(str)
s._df['sort_month'] = s._df['month'].apply(lambda x: {'10':1, '11':2, '12':3, '1':4, '2':5, '3':6, '4':7, '5':8, '6':9, '7':10, '8':11, '9':12}[x])

In [ ]:
df_sorted = s._df.sort_values(by=['sort_month', 'Water Year'])

In [ ]:
df_sorted[df_sorted['Water Year'] == 1922]

In [ ]:
# s._df['dateTime'] = pd.to_datetime(s._df['dateTime'])
s._df['day'] = s._df['Date'].dt.strftime('%d')
s._df['month'] = s._df['Date'].dt.strftime('%m')

# s._df['date'] = s._df['dateTime'].dt.strftime('%Y-%m-%d')
# s._df['dateTime'] = pd.to_datetime(s._df['dateTime'], format='%m-%d-%Y')

In [ ]:
s._df

In [ ]:
# s._df['date'] = pd.to_datetime(s._df['date'])

water_year_categories = pd.CategoricalDtype(
    categories=['10', '11', '12', '01', '02', '03', '04', '05', '06', '07', '08', '09'],
    ordered=True)

s._df['water_month'] = s._df['Date'].dt.month.astype(str).astype(water_year_categories)

In [ ]:
df_sorted = s._df.sort_values(by=['water_month', 'Water Year'])
df_sorted.to_csv('df_sorted.csv')

In [ ]:
df_sorted

In [ ]:
# df_sorted.pivot_table()
pivot = df_sorted.pivot(index="month-day", columns='Water Year', values='value')

In [ ]:
pivot

In [ ]:
c = pd.Categorical(['a', 'b', 'c', 'a', 'b', 'c'], ordered=True,
                   categories=['c', 'b', 'a'])

In [ ]:
c.min()

In [ ]:
c = pd.Categorical(['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', "Jun", 'Jul', 'Aug'], ordered=True,
#                    categories=['c', 'b', 'a']
                  )

In [ ]:
c.max()

In [ ]:
d = {'Date': [10,11,12,1,2,3,4,5,6,7,8,9], 'value':[123,3453,7646,2312,231,7674,2345,7456,2342,8677,2257,9374]}

In [ ]:
d= pd.DataFrame(d)

In [ ]:
d